<a href="https://colab.research.google.com/github/RajeshworM/IMD_GRID_DATA_EXTRACTION/blob/master/weather_IMD_dist_Update.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ============================================================
# DAILY TO MONTHLY DISTRICT DATA: RAINFALL, TMIN AND TMAX
# Start period: January 2025
# Input: Excel (.xlsx/.xls) or CSV files
# Output: One Excel workbook with combined and separate sheets
# ============================================================

!pip -q install openpyxl xlrd

import os
import re
import calendar
import numpy as np
import pandas as pd
from google.colab import files

# -------------------- 1. UPLOAD FILES -------------------------

print("Upload your daily rainfall, minimum temperature, and")
print("maximum temperature files together.")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No files uploaded.")

file_names = list(uploaded.keys())

print("\nUploaded files:")
for i, name in enumerate(file_names, 1):
    print(f"{i}. {name}")

# Automatically identify file types from filenames.
# If a filename is not recognised, select it manually.

def identify_variable(filename):
    name = filename.lower().replace(" ", "_")

    if any(x in name for x in [
        "rain", "precip", "prcp", "precipitation"
    ]):
        return "rainfall"

    if any(x in name for x in [
        "tmin", "temp_min", "minimum_temp", "min_temp",
        "mintemp", "tasmin", "minimum_temperature"
    ]):
        return "tmin"

    if any(x in name for x in [
        "tmax", "temp_max", "maximum_temp", "max_temp",
        "maxtemp", "tasmax", "maximum_temperature"
    ]):
        return "tmax"

    return None

file_map = {}

for name in file_names:
    variable = identify_variable(name)

    if variable and variable not in file_map:
        file_map[variable] = name

for variable in ["rainfall", "tmin", "tmax"]:
    if variable not in file_map:
        print(f"\nSelect the file for {variable}:")
        for i, name in enumerate(file_names, 1):
            print(f"{i}. {name}")

        choice = int(input("Enter file number: "))
        if choice < 1 or choice > len(file_names):
            raise ValueError("Invalid file number.")

        file_map[variable] = file_names[choice - 1]

if len(set(file_map.values())) != 3:
    raise ValueError(
        "Please upload three separate files, one for each variable."
    )

print("\nFile assignments:")
for variable, name in file_map.items():
    print(variable, ":", name)

# -------------------- 2. READ FILE ----------------------------

def read_raw_file(filename):
    extension = os.path.splitext(filename)[1].lower()

    if extension == ".csv":
        try:
            return pd.read_csv(
                filename, header=None, dtype=object,
                encoding="utf-8-sig"
            )
        except UnicodeDecodeError:
            return pd.read_csv(
                filename, header=None, dtype=object,
                encoding="latin1"
            )

    if extension in [".xlsx", ".xls"]:
        # First worksheet; read original header rows unchanged.
        return pd.read_excel(
            filename, header=None, dtype=object
        )

    raise ValueError(f"Unsupported file type: {filename}")

# -------------------- 3. DETECT DATE HEADER -------------------

def is_date_header(value):
    if pd.isna(value):
        return False

    text = str(value).strip().lower()
    return text in ["date", "dates", "time", "datetime", "day"]

def prepare_daily_data(filename, variable):
    raw = read_raw_file(filename)

    if raw.empty:
        raise ValueError(f"{filename} is empty.")

    # Find the row containing the date column.
    header_row = None

    for i in range(min(len(raw), 15)):
        first = raw.iloc[i, 0]
        if is_date_header(first):
            header_row = i
            break

    # If no explicit date label exists, find a row whose first
    # cell looks like a date, such as 1/1/2025.
    if header_row is None:
        for i in range(min(len(raw), 15)):
            value = raw.iloc[i, 0]
            if pd.isna(value):
                continue

            parsed = pd.to_datetime(
                str(value), errors="coerce"
            )
            if not pd.isna(parsed):
                header_row = i
                break

    if header_row is None:
        raise ValueError(
            f"Could not identify the date header in {filename}."
        )

    # Use the date-header row as column names.
    headers = []
    for j, value in enumerate(raw.iloc[header_row].tolist()):
        if j == 0:
            headers.append("date")
        else:
            headers.append(
                str(value).strip()
                if pd.notna(value) and str(value).strip()
                else f"district_{j}"
            )

    df = raw.iloc[header_row + 1:].copy()
    df.columns = headers
    df = df.loc[:, ~pd.Index(df.columns).duplicated()]
    df = df.dropna(how="all")

    # Parse dates. The example uses M/D/YYYY.
    df["date"] = pd.to_datetime(
        df["date"], errors="coerce"
    )

    df = df.dropna(subset=["date"])

    # Keep January 2025 onward.
    df = df[df["date"] >= pd.Timestamp("2025-01-01")].copy()

    if df.empty:
        raise ValueError(
            f"No valid dates from January 2025 onward in {filename}."
        )

    # District columns become numeric.
    district_columns = [
        c for c in df.columns if c != "date"
    ]

    for col in district_columns:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(",", "", regex=False)
                     .str.strip().replace(
                         {"": np.nan, "nan": np.nan, "NA": np.nan,
                          "N/A": np.nan, "-": np.nan}
                     ),
            errors="coerce"
        )

    # Convert from district-wide format to long format.
    long_df = df.melt(
        id_vars="date",
        value_vars=district_columns,
        var_name="District",
        value_name=variable
    )

    long_df["District"] = (
        long_df["District"].astype(str).str.strip()
    )

    long_df = long_df[
        long_df["District"].notna()
        & (long_df["District"] != "")
    ]

    # Do not treat missing observations as zero.
    long_df = long_df.dropna(subset=[variable])

    # Infer state name from the first header row, if available.
    # In your example this row contains repeated state names.
    state = "Unknown"
    for i in range(header_row):
        values = raw.iloc[i, 1:].dropna().astype(str).str.strip()
        values = [
            x for x in values
            if x and x.lower() not in ["nan", "none"]
        ]
        if values:
            counts = pd.Series(values).value_counts()
            if len(counts):
                candidate = counts.index[0]
                # Avoid interpreting dates or generic labels as states.
                if not is_date_header(candidate):
                    state = candidate

    long_df["State"] = state

    # If the file contains a state label only in its filename or
    # no state header, the state can be edited after processing.
    return long_df[["State", "District", "date", variable]]

# -------------------- 4. PROCESS THREE FILES ------------------

processed = {}

for variable, filename in file_map.items():
    print(f"\nProcessing {variable}: {filename}")
    processed[variable] = prepare_daily_data(filename, variable)
    print("Daily records:", len(processed[variable]))
    print("Districts:", processed[variable]["District"].nunique())
    print(
        "Date range:",
        processed[variable]["date"].min().date(),
        "to",
        processed[variable]["date"].max().date()
    )

# -------------------- 5. MONTHLY AGGREGATION ------------------

def monthly_aggregate(df, variable):
    x = df.copy()
    x["MonthDate"] = x["date"].dt.to_period("M").dt.to_timestamp()

    group_cols = ["State", "District", "MonthDate"]

    if variable == "rainfall":
        # Monthly rainfall = sum of daily rainfall.
        result = x.groupby(
            group_cols, as_index=False
        )[variable].sum(min_count=1)

    else:
        # Monthly temperature = average of daily temperatures.
        result = x.groupby(
            group_cols, as_index=False
        )[variable].mean()

    return result

monthly = {}

for variable, daily_df in processed.items():
    monthly[variable] = monthly_aggregate(daily_df, variable)

# Combine the three variables by state, district and month.
combined = None

for variable in ["rainfall", "tmin", "tmax"]:
    part = monthly[variable]

    if combined is None:
        combined = part
    else:
        combined = combined.merge(
            part,
            on=["State", "District", "MonthDate"],
            how="outer",
            validate="one_to_one"
        )

combined = combined.sort_values(
    ["State", "District", "MonthDate"]
).reset_index(drop=True)

combined["Year"] = combined["MonthDate"].dt.year
combined["Month"] = combined["MonthDate"].dt.month
combined["MonthName"] = combined["MonthDate"].dt.strftime("%b")

combined = combined[
    ["State", "District", "MonthDate", "Year",
     "Month", "MonthName", "rainfall", "tmin", "tmax"]
]

# -------------------- 6. CREATE WIDE MONTHLY SHEETS ------------

def make_wide(df, variable):
    x = df.copy()
    x["MonthLabel"] = x["MonthDate"].dt.strftime("%Y_%b").str.lower()

    wide = x.pivot_table(
        index=["State", "District"],
        columns="MonthLabel",
        values=variable,
        aggfunc="first"
    ).reset_index()

    wide.columns.name = None
    return wide

wide_rainfall = make_wide(monthly["rainfall"], "rainfall")
wide_tmin = make_wide(monthly["tmin"], "tmin")
wide_tmax = make_wide(monthly["tmax"], "tmax")

# -------------------- 7. SAVE EXCEL WORKBOOK ------------------

output_file = "Monthly_District_Weather_From_Jan2025.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    combined.to_excel(
        writer, sheet_name="Combined_Long", index=False
    )
    monthly["rainfall"].to_excel(
        writer, sheet_name="Rainfall_Long", index=False
    )
    monthly["tmin"].to_excel(
        writer, sheet_name="Tmin_Long", index=False
    )
    monthly["tmax"].to_excel(
        writer, sheet_name="Tmax_Long", index=False
    )
    wide_rainfall.to_excel(
        writer, sheet_name="Rainfall_Wide", index=False
    )
    wide_tmin.to_excel(
        writer, sheet_name="Tmin_Wide", index=False
    )
    wide_tmax.to_excel(
        writer, sheet_name="Tmax_Wide", index=False
    )

    # Improve date display and column widths.
    for ws in writer.book.worksheets:
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

        for column_cells in ws.columns:
            max_length = 0
            col_letter = column_cells[0].column_letter

            for cell in column_cells[:200]:
                if cell.value is not None:
                    max_length = max(
                        max_length, len(str(cell.value))
                    )

            ws.column_dimensions[col_letter].width = min(
                max(max_length + 2, 12), 25
            )

        # Display month dates as YYYY-MM-DD.
        for cell in ws["C"][1:]:
            if ws.title == "Combined_Long":
                cell.number_format = "yyyy-mm-dd"

print("\n==============================================")
print("MONTHLY CONVERSION COMPLETED")
print("Output:", output_file)
print("Combined monthly records:", len(combined))
print("Districts:", combined["District"].nunique())
print("Months:", combined["MonthDate"].nunique())
print("==============================================")

display(combined.head(15))

# -------------------- 8. DOWNLOAD RESULT ----------------------

files.download(output_file)


Upload your daily rainfall, minimum temperature, and
maximum temperature files together.


Saving IMD_Maximum_Temperature_0.5deg_Districtwise_try.csv to IMD_Maximum_Temperature_0.5deg_Districtwise_try.csv
Saving IMD_Minimum_Temperature_0.5deg_Districtwise_try.csv to IMD_Minimum_Temperature_0.5deg_Districtwise_try.csv
Saving IMD_Rainfall_Districtwise_2025_2026_try.csv to IMD_Rainfall_Districtwise_2025_2026_try.csv

Uploaded files:
1. IMD_Maximum_Temperature_0.5deg_Districtwise_try.csv
2. IMD_Minimum_Temperature_0.5deg_Districtwise_try.csv
3. IMD_Rainfall_Districtwise_2025_2026_try.csv

File assignments:
tmax : IMD_Maximum_Temperature_0.5deg_Districtwise_try.csv
tmin : IMD_Minimum_Temperature_0.5deg_Districtwise_try.csv
rainfall : IMD_Rainfall_Districtwise_2025_2026_try.csv

Processing tmax: IMD_Maximum_Temperature_0.5deg_Districtwise_try.csv
Daily records: 12654
Districts: 19
Date range: 2025-01-01 to 2026-10-28

Processing tmin: IMD_Minimum_Temperature_0.5deg_Districtwise_try.csv
Daily records: 12673
Districts: 19
Date range: 2025-01-01 to 2026-10-29

Processing rainfall: IM

/tmp/ipykernel_3736/1475469633.py:195: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .str.strip().replace(
/tmp/ipykernel_3736/1475469633.py:195: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .str.strip().replace(
/tmp/ipykernel_3736/1475469633.py:195: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .str.

Daily records: 12673
Districts: 19
Date range: 2025-01-01 to 2026-12-01

MONTHLY CONVERSION COMPLETED
Output: Monthly_District_Weather_From_Jan2025.xlsx
Combined monthly records: 437
Districts: 19
Months: 23


,State,District,MonthDate,Year,Month,MonthName,rainfall,tmin,tmax
0,Madhya Pradesh,Ashoknagar,2025-01-01,2025,1,Jan,0.000000,14.594743,31.292587
1,Madhya Pradesh,Ashoknagar,2025-02-01,2025,2,Feb,0.000000,16.756219,35.257319
2,Madhya Pradesh,Ashoknagar,2025-03-01,2025,3,Mar,0.000000,20.490173,39.018939
3,Madhya Pradesh,Ashoknagar,2025-04-01,2025,4,Apr,2.380748,24.835089,41.636266
4,Madhya Pradesh,Ashoknagar,2025-05-01,2025,5,May,34.512807,25.130302,37.845510
5,Madhya Pradesh,Ashoknagar,2025-06-01,2025,6,Jun,398.621350,25.176447,35.545290
6,Madhya Pradesh,Ashoknagar,2025-07-01,2025,7,Jul,713.333328,23.602536,30.332798
7,Madhya Pradesh,Ashoknagar,2025-08-01,2025,8,Aug,260.046134,23.471050,31.333501
8,Madhya Pradesh,Ashoknagar,2025-09-01,2025,9,Sep,78.018721,23.068717,31.538835
9,Madhya Pradesh,Ashoknagar,2025-10-01,2025,10,Oct,69.852728,21.677418,32.535778


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>